In [ ]:
!pip install -q kaggle tensorflow matplotlib scikit-learn

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d mahendarbyra/behsof-dataset -p /content
!unzip -q "/content/behsof-dataset.zip" -d "/content/behsof"

In [ ]:
import os

SOURCE_DIR = "/content/behsof/BEHSOF"
SPLIT_DIR = "/content/behsof_split"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42
EPOCHS_STAGE1 = 20
EPOCHS_STAGE2 = 10

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

classes = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))]

for split in ["train", "val", "test"]:
    for c in classes:
        os.makedirs(os.path.join(SPLIT_DIR, split, c), exist_ok=True)

for c in classes:
    files = [
        os.path.join(SOURCE_DIR, c, f)
        for f in os.listdir(os.path.join(SOURCE_DIR, c))
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    ]

    train_files, temp_files = train_test_split(
        files,
        test_size=0.30,
        random_state=SEED,
        shuffle=True
    )

    val_files, test_files = train_test_split(
        temp_files,
        test_size=0.50,
        random_state=SEED,
        shuffle=True
    )

    for f in train_files:
        shutil.copy2(f, os.path.join(SPLIT_DIR, "train", c, os.path.basename(f)))

    for f in val_files:
        shutil.copy2(f, os.path.join(SPLIT_DIR, "val", c, os.path.basename(f)))

    for f in test_files:
        shutil.copy2(f, os.path.join(SPLIT_DIR, "test", c, os.path.basename(f)))

print("Split complete.")

In [ ]:
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    for c in classes:
        folder = os.path.join(SPLIT_DIR, split, c)
        count = len([
            f for f in os.listdir(folder)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
        ])
        print(c, ":", count)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_ds = image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

test_ds = image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
train_ds = train_ds.prefetch(1)
val_ds = val_ds.prefetch(1)
test_ds = test_ds.prefetch(1)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
class_counts = {}
for _, labels in train_ds.unbatch():
    label = int(labels.numpy())
    class_counts[label] = class_counts.get(label, 0) + 1

print("Class counts:", class_counts)

total = sum(class_counts.values())
class_weight = {
    cls: total / (len(class_counts) * count)
    for cls, count in class_counts.items()
}

print("Class weights:", class_weight)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.20),
])

base = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callbacks_stage1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=2,
        factor=0.5
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "fatty_liver_checkpoint_stage1.keras",
        save_best_only=True,
        monitor="val_loss"
    )
]

In [ ]:
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks_stage1,
    class_weight=class_weight
)

In [ ]:
print("Stage 1 epochs actually run:", len(history_stage1.history["loss"]))
print(history_stage1.history.keys())

In [ ]:
base.trainable = True

for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callbacks_stage2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=2,
        factor=0.5
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "fatty_liver_checkpoint_stage2.keras",
        save_best_only=True,
        monitor="val_loss"
    )
]

In [ ]:
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks_stage2,
    class_weight=class_weight
)

In [ ]:
print("Stage 2 epochs actually run:", len(history_stage2.history["loss"]))
print(history_stage2.history.keys())

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_stage1.history["accuracy"], label="stage1_train_accuracy")
plt.plot(history_stage1.history["val_accuracy"], label="stage1_val_accuracy")
plt.title("Fatty Liver Model Accuracy - Stage 1")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_stage1.history["loss"], label="stage1_train_loss")
plt.plot(history_stage1.history["val_loss"], label="stage1_val_loss")
plt.title("Fatty Liver Model Loss - Stage 1")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_stage1.history["loss"], label="stage1_train_loss")
plt.plot(history_stage1.history["val_loss"], label="stage1_val_loss")
plt.title("Fatty Liver Model Loss - Stage 1")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_stage2.history["loss"], label="stage2_train_loss")
plt.plot(history_stage2.history["val_loss"], label="stage2_val_loss")
plt.title("Fatty Liver Model Loss - Stage 2 Fine-Tuning")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)

print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(pred_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=class_names))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))
print("Macro F1 Score:", f1_score(y_true, y_pred, average="macro"))

In [ ]:
plt.figure(figsize=(14, 14))

for images, labels in test_ds.take(1):
    preds = model.predict(images, verbose=0)

    nafld_idx = class_names.index("NAFLD")
    non_nafld_idx = class_names.index("Non-NAFLD")

    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        nafld_prob = float(preds[i][nafld_idx])
        non_nafld_prob = float(preds[i][non_nafld_idx])
        margin = abs(nafld_prob - non_nafld_prob)

        if margin < 0.15:
            result = "Uncertain"
            pred_name = class_names[int(np.argmax(preds[i]))]
            confidence = max(nafld_prob, non_nafld_prob) * 100
        elif nafld_prob > non_nafld_prob:
            result = "Fatty Liver Detected"
            pred_name = "NAFLD"
            confidence = nafld_prob * 100
        else:
            result = "No Fatty Liver Detected"
            pred_name = "Non-NAFLD"
            confidence = non_nafld_prob * 100

        plt.title(
            f"Result: {result}\nPredicted Type: {pred_name}\nConfidence: {confidence:.1f}%",
            fontsize=10
        )
        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image

uploaded = files.upload()
img_path = list(uploaded.keys())[0]

img = image.load_img(img_path, target_size=IMG_SIZE)
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array, verbose=0)[0]

nafld_idx = class_names.index("NAFLD")
non_nafld_idx = class_names.index("Non-NAFLD")

nafld_prob = float(prediction[nafld_idx])
non_nafld_prob = float(prediction[non_nafld_idx])
margin = abs(nafld_prob - non_nafld_prob)

if margin < 0.15:
    result = "Uncertain"
    pred_class_name = class_names[int(np.argmax(prediction))]
    confidence = max(nafld_prob, non_nafld_prob) * 100
elif nafld_prob > non_nafld_prob:
    result = "Fatty Liver Detected"
    pred_class_name = "NAFLD"
    confidence = nafld_prob * 100
else:
    result = "No Fatty Liver Detected"
    pred_class_name = "Non-NAFLD"
    confidence = non_nafld_prob * 100

plt.figure(figsize=(6, 6))
plt.imshow(image.load_img(img_path))
plt.title(
    f"Result: {result}\nPredicted Type: {pred_class_name}\nConfidence: {confidence:.1f}%",
    fontsize=12
)
plt.axis("off")
plt.show()

print(f"Result: {result}")
print(f"Predicted Type: {pred_class_name}")
print(f"Confidence: {confidence:.1f}%")
print("Raw Probabilities:", prediction)
print("Decision Margin:", round(margin, 4))

In [ ]:
model.save("fatty_liver_model_224_finetuned.keras")
print("Model saved successfully")